In [2]:
from pathlib import Path
import pandas as pd

# ------------------------------------------------------------
# Expected experiment structure
# ------------------------------------------------------------

QUICK_SWEEP_ROOTS = {
    "splitnet_attn": Path("recent_analysis/physics_tests"),
    "splitnet": Path("recent_analysis_split_na/physics_tests"),
    "attn_unet": Path("recent_analysis_unet/physics_tests"),
    "unet": Path("recent_analysis_unet_na/physics_tests"),
    "prof_unet": Path("recent_analysis_prof_unet/physics_tests"),
}

QUICK_DATASET_MODES = ["border", "border_pressure", "fixed"]

QUICK_DARCY_WEIGHTS = [
    0.0,
    0.001,
    0.01,
    0.1,
    1.0,
    10.0,
]


OFFICIAL_DARCY_ROOT = Path("official_darcy/final")

OFFICIAL_DARCY_MODEL_TYPES = [
    "splitnet_attn",
    "prof_unet",
    "attn_unet",
    "splitnet",
    "unet",
]

OFFICIAL_DARCY_DATASET_MODES = ["fixed", "border"]

OFFICIAL_DARCY_WEIGHTS = [
    5.0,
    0.1,
    1.0,
    10.0,
]


BASELINE_FULL_ROOT = Path("baseline_full/final")

BASELINE_FULL_MODEL_TYPES = [
    "splitnet_attn",
    "attn_unet",
    "prof_unet",
]

BASELINE_FULL_DATASET_MODES = [
    "fixed",
    "border",
]


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def safe_weight(w):
    return str(w).replace(".", "p")


def check_file(path):
    return Path(path).exists()


def add_expected(rows, category, model_type, dataset_mode, training_mode, darcy_weight, base_path, run_name):
    base_path = Path(base_path)

    expected = {
        "category": category,
        "model_type": model_type,
        "dataset_mode": dataset_mode,
        "training_mode": training_mode,
        "darcy_weight": darcy_weight,
        "run_name": run_name,

        "best_state_path": str(base_path / f"{run_name}_best_state.pt"),
        "final_state_path": str(base_path / f"{run_name}_final_state.pt"),
        "history_csv_path": str(base_path / f"{run_name}_history.csv"),
        "history_pt_path": str(base_path / f"{run_name}_history.pt"),
        "summary_json_path": str(base_path / f"{run_name}_summary.json"),
    }

    expected["has_best_state"] = check_file(expected["best_state_path"])
    expected["has_final_state"] = check_file(expected["final_state_path"])
    expected["has_history_csv"] = check_file(expected["history_csv_path"])
    expected["has_history_pt"] = check_file(expected["history_pt_path"])
    expected["has_summary_json"] = check_file(expected["summary_json_path"])

    expected["complete_core"] = (
        expected["has_best_state"]
        and expected["has_history_csv"]
    )

    rows.append(expected)


# ------------------------------------------------------------
# Build expected file table
# ------------------------------------------------------------

rows = []

# 1. Quick thin physics-limited Darcy sweeps
for model_type, root in QUICK_SWEEP_ROOTS.items():
    for dataset_mode in QUICK_DATASET_MODES:
        for w in QUICK_DARCY_WEIGHTS:
            sw = safe_weight(w)
            run_name = f"{dataset_mode}_physics_limited_{model_type}_darcy_{sw}"
            base_path = root / dataset_mode

            add_expected(
                rows=rows,
                category="quick_thin_darcy_sweep",
                model_type=model_type,
                dataset_mode=dataset_mode,
                training_mode="physics_limited",
                darcy_weight=w,
                base_path=base_path,
                run_name=run_name,
            )

# 2. Official dense/longer Darcy models
for model_type in OFFICIAL_DARCY_MODEL_TYPES:
    for dataset_mode in OFFICIAL_DARCY_DATASET_MODES:
        for w in OFFICIAL_DARCY_WEIGHTS:
            sw = safe_weight(w)
            run_name = f"{dataset_mode}_physics_limited_{model_type}_darcy_{sw}"

            add_expected(
                rows=rows,
                category="official_darcy",
                model_type=model_type,
                dataset_mode=dataset_mode,
                training_mode="physics_limited",
                darcy_weight=w,
                base_path=OFFICIAL_DARCY_ROOT,
                run_name=run_name,
            )

# 3. Baseline full no-Darcy models
for model_type in BASELINE_FULL_MODEL_TYPES:
    for dataset_mode in BASELINE_FULL_DATASET_MODES:
        run_name = f"{dataset_mode}_{model_type}_baseline_full_nodarcy"

        add_expected(
            rows=rows,
            category="baseline_full_nodarcy",
            model_type=model_type,
            dataset_mode=dataset_mode,
            training_mode="baseline_full",
            darcy_weight=0.0,
            base_path=BASELINE_FULL_ROOT,
            run_name=run_name,
        )

df = pd.DataFrame(rows)

# ------------------------------------------------------------
# Print useful summaries
# ------------------------------------------------------------

print("Total expected runs:", len(df))
print("Complete core runs:", df["complete_core"].sum())
print("Missing/incomplete runs:", (~df["complete_core"]).sum())

print("\nBy category:")
print(df.groupby("category")["complete_core"].agg(["sum", "count"]))

print("\nMissing core files:")
missing = df[~df["complete_core"]].copy()

if len(missing) == 0:
    print("None. All expected core files were found.")
else:
    display_cols = [
        "category",
        "model_type",
        "dataset_mode",
        "training_mode",
        "darcy_weight",
        "run_name",
        "has_best_state",
        "has_history_csv",
        "best_state_path",
    ]
    print(missing[display_cols].to_string(index=False))

# Save the full table
out_path = "model_file_inventory.csv"
df.to_csv(out_path, index=False)
print(f"\nSaved inventory to: {out_path}")

Total expected runs: 136
Complete core runs: 136
Missing/incomplete runs: 0

By category:
                        sum  count
category                          
baseline_full_nodarcy     6      6
official_darcy           40     40
quick_thin_darcy_sweep   90     90

Missing core files:
None. All expected core files were found.

Saved inventory to: model_file_inventory.csv


In [2]:
from pathlib import Path
import pandas as pd

# ------------------------------------------------------------
# Expected experiment structure
# ------------------------------------------------------------

TEST_OUTPUT_ROOT = Path("test_eval_outputs")

QUICK_SWEEP_RESULTS_DIR = TEST_OUTPUT_ROOT / "quick_sweeps"
OFFICIAL_DARCY_RESULTS_DIR = TEST_OUTPUT_ROOT / "official_darcy"
BASELINE_FULL_RESULTS_DIR = TEST_OUTPUT_ROOT / "baseline_full"
DATA_AMOUNT_RESULTS_DIR = TEST_OUTPUT_ROOT / "data_amount_tests"

QUICK_SWEEP_ROOTS = {
    "splitnet_attn": Path("recent_analysis/physics_tests"),
    "splitnet": Path("recent_analysis_split_na/physics_tests"),
    "attn_unet": Path("recent_analysis_unet/physics_tests"),
    "unet": Path("recent_analysis_unet_na/physics_tests"),
    "prof_unet": Path("recent_analysis_prof_unet/physics_tests"),
}

QUICK_DATASET_MODES = ["border", "border_pressure", "fixed"]
QUICK_DARCY_WEIGHTS = [0.0, 0.001, 0.01, 0.1, 1.0, 10.0]

OFFICIAL_DARCY_MODEL_TYPES = [
    "splitnet_attn",
    "prof_unet",
    "attn_unet",
    "splitnet",
    "unet",
]

OFFICIAL_DARCY_DATASET_MODES = ["fixed", "border"]
OFFICIAL_DARCY_WEIGHTS = [5.0, 0.1, 1.0, 10.0]

BASELINE_FULL_MODEL_TYPES = [
    "splitnet_attn",
    "attn_unet",
    "prof_unet",
]

BASELINE_FULL_DATASET_MODES = ["fixed", "border"]

# Optional: data amount tests
SIM_AMOUNTS = [25, 50, 100]
DATA_AMOUNT_DATASET_MODES = ["fixed"]
DATA_AMOUNT_PHYSICS_MODELS = {
    "splitnet_attn": [1.0, 10.0],
    "attn_unet": [1.0, 10.0],
    "prof_unet": [1.0, 10.0],
}
DATA_AMOUNT_BASELINE_MODELS = [
    "splitnet_attn",
    "attn_unet",
    "prof_unet",
]


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def safe_weight(w):
    return str(w).replace(".", "p")


def check_file(path):
    return Path(path).exists()


def add_expected_test(rows, category, model_type, dataset_mode, training_mode,
                      darcy_weight, results_dir, eval_name):
    results_dir = Path(results_dir)

    row = {
        "category": category,
        "model_type": model_type,
        "dataset_mode": dataset_mode,
        "training_mode": training_mode,
        "darcy_weight": darcy_weight,
        "eval_name": eval_name,

        "npz_path": str(results_dir / f"eval_{eval_name}.npz"),
        "summary_json_path": str(results_dir / f"eval_{eval_name}_summary.json"),
    }

    row["has_npz"] = check_file(row["npz_path"])
    row["has_summary_json"] = check_file(row["summary_json_path"])

    row["complete_test"] = row["has_npz"] and row["has_summary_json"]

    rows.append(row)


# ------------------------------------------------------------
# Build expected test output table
# ------------------------------------------------------------

rows = []

# 1. Quick thin Darcy sweep test outputs
for model_type in QUICK_SWEEP_ROOTS.keys():
    for dataset_mode in QUICK_DATASET_MODES:
        for w in QUICK_DARCY_WEIGHTS:
            sw = safe_weight(w)
            run_name = f"{dataset_mode}_physics_limited_{model_type}_darcy_{sw}"
            eval_name = f"quick_{run_name}"

            add_expected_test(
                rows,
                category="quick_thin_darcy_sweep",
                model_type=model_type,
                dataset_mode=dataset_mode,
                training_mode="physics_limited",
                darcy_weight=w,
                results_dir=QUICK_SWEEP_RESULTS_DIR,
                eval_name=eval_name,
            )

# 2. Official Darcy test outputs
for model_type in OFFICIAL_DARCY_MODEL_TYPES:
    for dataset_mode in OFFICIAL_DARCY_DATASET_MODES:
        for w in OFFICIAL_DARCY_WEIGHTS:
            sw = safe_weight(w)
            run_name = f"{dataset_mode}_physics_limited_{model_type}_darcy_{sw}"
            eval_name = f"official_{run_name}"

            add_expected_test(
                rows,
                category="official_darcy",
                model_type=model_type,
                dataset_mode=dataset_mode,
                training_mode="physics_limited",
                darcy_weight=w,
                results_dir=OFFICIAL_DARCY_RESULTS_DIR,
                eval_name=eval_name,
            )

# 3. Baseline full no-Darcy test outputs
for model_type in BASELINE_FULL_MODEL_TYPES:
    for dataset_mode in BASELINE_FULL_DATASET_MODES:
        run_name = f"{dataset_mode}_{model_type}_baseline_full_nodarcy"
        eval_name = f"baseline_{run_name}"

        add_expected_test(
            rows,
            category="baseline_full_nodarcy",
            model_type=model_type,
            dataset_mode=dataset_mode,
            training_mode="baseline_full",
            darcy_weight=0.0,
            results_dir=BASELINE_FULL_RESULTS_DIR,
            eval_name=eval_name,
        )

# 4. Data amount test outputs
for sim_max in SIM_AMOUNTS:
    for dataset_mode in DATA_AMOUNT_DATASET_MODES:

        for model_type, weights in DATA_AMOUNT_PHYSICS_MODELS.items():
            for w in weights:
                sw = safe_weight(w)
                run_name = (
                    f"{dataset_mode}_{model_type}_"
                    f"physics_limited_darcy_{sw}_sims_{sim_max}"
                )
                eval_name = f"data_amount_{run_name}"

                add_expected_test(
                    rows,
                    category="data_amount_physics_limited",
                    model_type=model_type,
                    dataset_mode=dataset_mode,
                    training_mode="physics_limited",
                    darcy_weight=w,
                    results_dir=DATA_AMOUNT_RESULTS_DIR,
                    eval_name=eval_name,
                )

        for model_type in DATA_AMOUNT_BASELINE_MODELS:
            run_name = (
                f"{dataset_mode}_{model_type}_"
                f"baseline_full_nodarcy_sims_{sim_max}"
            )
            eval_name = f"data_amount_{run_name}"

            add_expected_test(
                rows,
                category="data_amount_baseline_full",
                model_type=model_type,
                dataset_mode=dataset_mode,
                training_mode="baseline_full",
                darcy_weight=0.0,
                results_dir=DATA_AMOUNT_RESULTS_DIR,
                eval_name=eval_name,
            )


df = pd.DataFrame(rows)

# ------------------------------------------------------------
# Print useful summaries
# ------------------------------------------------------------

print("Total expected test runs:", len(df))
print("Complete test runs:", df["complete_test"].sum())
print("Missing/incomplete test runs:", (~df["complete_test"]).sum())

print("\nBy category:")
print(df.groupby("category")["complete_test"].agg(["sum", "count"]))

print("\nMissing test files:")
missing = df[~df["complete_test"]].copy()

if len(missing) == 0:
    print("None. All expected test files were found.")
else:
    display_cols = [
        "category",
        "model_type",
        "dataset_mode",
        "training_mode",
        "darcy_weight",
        "eval_name",
        "has_npz",
        "has_summary_json",
        "npz_path",
        "summary_json_path",
    ]
    print(missing[display_cols].to_string(index=False))

# Save full inventory
out_path = "test_file_inventory.csv"
df.to_csv(out_path, index=False)
print(f"\nSaved test inventory to: {out_path}")

Total expected test runs: 163
Complete test runs: 133
Missing/incomplete test runs: 30

By category:
                             sum  count
category                               
baseline_full_nodarcy          4      6
data_amount_baseline_full      0      9
data_amount_physics_limited    0     18
official_darcy                39     40
quick_thin_darcy_sweep        90     90

Missing test files:
                   category    model_type dataset_mode   training_mode  darcy_weight                                                           eval_name  has_npz  has_summary_json                                                                                                         npz_path                                                                                                         summary_json_path
             official_darcy     prof_unet       border physics_limited          10.0                official_border_physics_limited_prof_unet_darcy_10p0    False             False    